In [0]:
# Read YouTube API key from secret scope
YT_API_KEY = dbutils.secrets.get(scope="youtube-wrapped", key="yt_api_key")
print(f"✅ Key loaded (first 8 chars): {YT_API_KEY[:8]}...")

In [0]:
# ============================================================
# ENRICHMENT: YouTube Data API + MusicBrainz
# ============================================================
# Fetch video metadata (category, duration, tags, views) for every 
# unique video_id, and genre data for every unique music artist.
# Cache results in Delta tables for idempotent reruns.
# ============================================================

import requests
import time
from typing import List, Dict
from pyspark.sql.functions import col, lit, current_timestamp, explode
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, ArrayType, TimestampType

SILVER_TABLE = "youtube_wrapped.silver.watch_history_clean"
ENRICH_VIDEOS_TABLE = "youtube_wrapped.silver.video_enrichment"
ENRICH_ARTISTS_TABLE = "youtube_wrapped.silver.artist_enrichment"

In [0]:
# Get unique video_ids from silver
df_silver = spark.table(SILVER_TABLE)

unique_video_ids = [row.video_id for row in df_silver.select("video_id").distinct().collect()]

print(f"Unique videos to potentially enrich: {len(unique_video_ids):,}")

# If enrichment table already exists, skip videos we've already fetched
try:
    already_enriched = {row.video_id for row in spark.table(ENRICH_VIDEOS_TABLE).select("video_id").collect()}
    print(f"Already enriched: {len(already_enriched):,}")
except Exception:
    already_enriched = set()
    print("No existing enrichment table — will create fresh")

to_fetch = [v for v in unique_video_ids if v not in already_enriched]
print(f"To fetch this run: {len(to_fetch):,}")

# Estimate quota usage
batches_needed = (len(to_fetch) + 49) // 50  # ceil division
print(f"API batches needed: {batches_needed} (= {batches_needed} quota units out of 10,000 daily)")

In [0]:
YT_VIDEOS_ENDPOINT = "https://www.googleapis.com/youtube/v3/videos"

# YouTube's category IDs mapped to human-readable names
# Reference: https://developers.google.com/youtube/v3/docs/videoCategories/list
CATEGORY_MAP = {
    "1": "Film & Animation", "2": "Autos & Vehicles", "10": "Music",
    "15": "Pets & Animals", "17": "Sports", "19": "Travel & Events",
    "20": "Gaming", "22": "People & Blogs", "23": "Comedy",
    "24": "Entertainment", "25": "News & Politics", "26": "Howto & Style",
    "27": "Education", "28": "Science & Technology", "29": "Nonprofits & Activism"
}

def fetch_video_batch(video_ids: List[str], api_key: str) -> List[Dict]:
    """Fetch metadata for up to 50 video IDs in one API call."""
    params = {
        "part": "snippet,contentDetails,statistics",
        "id": ",".join(video_ids),
        "key": api_key,
        "maxResults": 50
    }
    resp = requests.get(YT_VIDEOS_ENDPOINT, params=params, timeout=15)
    resp.raise_for_status()
    return resp.json().get("items", [])

def parse_iso8601_duration(duration: str) -> int:
    """Convert ISO 8601 duration (PT4M13S) to total seconds."""
    import re
    if not duration:
        return 0
    pattern = r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?"
    match = re.match(pattern, duration)
    if not match:
        return 0
    hours = int(match.group(1) or 0)
    minutes = int(match.group(2) or 0)
    seconds = int(match.group(3) or 0)
    return hours * 3600 + minutes * 60 + seconds

def normalize_video(item: Dict) -> Dict:
    """Flatten the YouTube API response into a clean dict."""
    snippet = item.get("snippet", {})
    content = item.get("contentDetails", {})
    stats = item.get("statistics", {})
    cat_id = snippet.get("categoryId", "")
    return {
        "video_id": item.get("id", ""),
        "category_id": cat_id,
        "category_name": CATEGORY_MAP.get(cat_id, "Unknown"),
        "default_audio_language": snippet.get("defaultAudioLanguage"),
        "duration_seconds": parse_iso8601_duration(content.get("duration", "")),
        "view_count": int(stats.get("viewCount", 0)) if stats.get("viewCount") else None,
        "like_count": int(stats.get("likeCount", 0)) if stats.get("likeCount") else None,
        "tags": snippet.get("tags", []),
        "published_at": snippet.get("publishedAt"),
    }

In [0]:
# Fetch in batches of 50
all_results = []
failed_batches = 0

for i in range(0, len(to_fetch), 50):
    batch = to_fetch[i:i+50]
    try:
        items = fetch_video_batch(batch, YT_API_KEY)
        all_results.extend([normalize_video(item) for item in items])
        print(f"Batch {i//50 + 1}/{batches_needed}: fetched {len(items)} videos")
    except Exception as e:
        failed_batches += 1
        print(f"⚠️ Batch {i//50 + 1} failed: {e}")
    time.sleep(0.1)  # be polite to the API

print(f"\n✅ Total videos enriched: {len(all_results):,}")
print(f"⚠️  Failed batches: {failed_batches}")

In [0]:
# Define schema explicitly for consistency
video_schema = StructType([
    StructField("video_id", StringType(), False),
    StructField("category_id", StringType(), True),
    StructField("category_name", StringType(), True),
    StructField("default_audio_language", StringType(), True),
    StructField("duration_seconds", IntegerType(), True),
    StructField("view_count", LongType(), True),
    StructField("like_count", LongType(), True),
    StructField("tags", ArrayType(StringType()), True),
    StructField("published_at", StringType(), True),
])

if all_results:
    df_enriched = spark.createDataFrame(all_results, schema=video_schema)
    df_enriched = df_enriched.withColumn("_enriched_at", current_timestamp())
    
    # Append mode — preserves prior enrichment runs
    mode = "append" if already_enriched else "overwrite"
    
    (
        df_enriched.write
        .mode(mode)
        .option("mergeSchema", "true")
        .saveAsTable(ENRICH_VIDEOS_TABLE)
    )
    print(f"✅ Wrote {df_enriched.count():,} enrichment rows to {ENRICH_VIDEOS_TABLE}")
else:
    print("No new enrichments to write")

In [0]:
%sql
-- Category breakdown of your watches
SELECT category_name, COUNT(*) AS unique_videos
FROM youtube_wrapped.silver.video_enrichment
GROUP BY category_name
ORDER BY unique_videos DESC;

-- Total watch time estimate (assuming you watched full videos)
SELECT 
  ROUND(SUM(duration_seconds) / 3600.0, 1) AS total_hours_available,
  ROUND(AVG(duration_seconds) / 60.0, 1) AS avg_video_minutes
FROM youtube_wrapped.silver.video_enrichment;

In [0]:
MB_BASE = "https://musicbrainz.org/ws/2"
MB_HEADERS = {
    # MusicBrainz requires a User-Agent identifying your app
    "User-Agent": "youtube-wrapped/0.1 (github.com/Shaan-alpha/youtube-wrapped)"
}

def search_artist_on_mb(artist_name: str) -> Dict:
    """Search MusicBrainz for an artist and return top match with tags/genres."""
    params = {"query": f'artist:"{artist_name}"', "fmt": "json", "limit": 1}
    
    try:
        resp = requests.get(f"{MB_BASE}/artist", params=params, headers=MB_HEADERS, timeout=15)
        resp.raise_for_status()
        data = resp.json()
        
        artists = data.get("artists", [])
        if not artists:
            return {"artist_name": artist_name, "mb_id": None, "genres": [], "tags": [], "country": None}
        
        top = artists[0]
        return {
            "artist_name": artist_name,
            "mb_id": top.get("id"),
            "mb_name": top.get("name"),
            "genres": [g.get("name") for g in top.get("genres", []) if g.get("name")],
            "tags": [t.get("name") for t in top.get("tags", []) if t.get("name")],
            "country": top.get("country"),
            "match_score": top.get("score"),
        }
    except Exception as e:
        return {"artist_name": artist_name, "mb_id": None, "genres": [], "tags": [], "country": None, "error": str(e)}

In [0]:
# Get unique artists from music-flagged rows
unique_artists = [
    row.artist_name_from_topic 
    for row in df_silver.filter(col("is_music_topic") == True)
                        .select("artist_name_from_topic")
                        .distinct()
                        .collect()
    if row.artist_name_from_topic
]

print(f"Unique artists to enrich: {len(unique_artists):,}")

# Skip already-enriched artists
try:
    already_enriched_artists = {row.artist_name for row in spark.table(ENRICH_ARTISTS_TABLE).select("artist_name").collect()}
except Exception:
    already_enriched_artists = set()

artists_to_fetch = [a for a in unique_artists if a not in already_enriched_artists]
print(f"Artists to fetch this run: {len(artists_to_fetch):,}")
print(f"Estimated time: {len(artists_to_fetch)} seconds (1 req/sec rate limit)")

In [0]:
artist_results = []

for idx, artist in enumerate(artists_to_fetch, start=1):
    result = search_artist_on_mb(artist)
    artist_results.append(result)
    
    if idx % 10 == 0:
        print(f"Progress: {idx}/{len(artists_to_fetch)} — latest: {artist} → genres: {result.get('genres', [])[:3]}")
    
    time.sleep(1.1)  # MusicBrainz: max 1 req/sec, add buffer

print(f"\n✅ Enriched {len(artist_results)} artists")

In [0]:
artist_schema = StructType([
    StructField("artist_name", StringType(), False),
    StructField("mb_id", StringType(), True),
    StructField("mb_name", StringType(), True),
    StructField("genres", ArrayType(StringType()), True),
    StructField("tags", ArrayType(StringType()), True),
    StructField("country", StringType(), True),
    StructField("match_score", IntegerType(), True),
])

if artist_results:
    # Keep only the fields in the schema
    clean_results = [
        {k: r.get(k) for k in ["artist_name", "mb_id", "mb_name", "genres", "tags", "country", "match_score"]}
        for r in artist_results
    ]
    
    df_artists = spark.createDataFrame(clean_results, schema=artist_schema)
    df_artists = df_artists.withColumn("_enriched_at", current_timestamp())
    
    mode = "append" if already_enriched_artists else "overwrite"
    
    (
        df_artists.write
        .mode(mode)
        .option("mergeSchema", "true")
        .saveAsTable(ENRICH_ARTISTS_TABLE)
    )
    print(f"✅ Wrote {df_artists.count():,} artist enrichment rows")

In [0]:
%sql
WITH artist_plays AS (
  SELECT 
    a.artist_name,
    explode(a.genres) AS genre,
    w.video_id
  FROM youtube_wrapped.silver.artist_enrichment a
  JOIN youtube_wrapped.silver.watch_history_clean w
    ON a.artist_name = w.artist_name_from_topic
  WHERE w.is_music_topic = true
)
SELECT 
  genre,
  COUNT(DISTINCT artist_name) AS artists,
  COUNT(*) AS total_plays
FROM artist_plays
GROUP BY genre
ORDER BY total_plays DESC
LIMIT 15;

In [0]:
%sql
-- How many artists have empty genres?
SELECT COUNT(*) AS unknown_genre_artists 
FROM youtube_wrapped.silver.artist_enrichment 
WHERE size(genres) = 0;

In [0]:
%sql
-- Sample of unknown artists (with tags as fallback signal)
SELECT 
  artist_name, 
  tags, 
  country
FROM youtube_wrapped.silver.artist_enrichment
WHERE size(genres) = 0
ORDER BY artist_name
LIMIT 20;

In [0]:
%sql
WITH exploded AS (
  SELECT 
    a.artist_name,
    explode(a.tags) AS raw_tag
  FROM youtube_wrapped.silver.artist_enrichment a
  WHERE a.tags IS NOT NULL AND size(a.tags) > 0
),
artist_tags AS (
  SELECT 
    e.artist_name,
    lower(trim(e.raw_tag)) AS tag,
    w.video_id
  FROM exploded e
  JOIN youtube_wrapped.silver.watch_history_clean w
    ON e.artist_name = w.artist_name_from_topic
  WHERE w.is_music_topic = true
),
genre_tags AS (
  SELECT *
  FROM artist_tags
  WHERE tag NOT IN (
      'indian', 'norwegian', 'norway', 'pakistani', 'american', 'british',
      'canadian', 'australian', 'japanese', 'korean', 'french', 'spanish',
      'mexican', 'german', 'irish', 'scottish', 'swedish',
      'female vocalists', 'male vocalists', 'singer-songwriter',
      'theatre', 'asian', 'world fusion', 'world',
      'academy award winner', 'cham cham', 'ar-rahman'
    )
    AND tag NOT RLIKE '^[0-9]{2,4}s?$'
    AND tag RLIKE '^[a-z0-9 \\-/&]+$'
)
SELECT 
  tag AS genre,
  COUNT(DISTINCT artist_name) AS artists,
  COUNT(*) AS total_plays
FROM genre_tags
GROUP BY tag
ORDER BY total_plays DESC
LIMIT 20;

In [0]:
%sql
SELECT 
  CASE 
    WHEN tags IS NULL OR size(tags) = 0 THEN 'no_tags'
    WHEN size(tags) BETWEEN 1 AND 5 THEN 'few_tags (1-5)'
    WHEN size(tags) BETWEEN 6 AND 20 THEN 'rich_tags (6-20)'
    ELSE 'very_rich (20+)'
  END AS tag_richness,
  COUNT(*) AS artists
FROM youtube_wrapped.silver.artist_enrichment
GROUP BY tag_richness
ORDER BY artists DESC;